# 01 — Hugging Face Audio Models in 10 Steps (Google Colab Edition)
**Cours : Internet et musique agentique (HEM)** — *Zéro installation locale / 100% Cloud GPU*

> **FR :** Activez le GPU gratuit dans Colab : **Exécution → Modifier le type d'exécution → GPU T4**.
>
> **EN:** Enable the free GPU in Colab: **Runtime → Change runtime type → T4 GPU**.

## Step 1. Environment Check (Colab GPU & Libraries) / Étape 1. Environnement Cloud

In [ ]:
!pip install -q transformers accelerate librosa soundfile torchaudio

import torch, transformers, librosa
from pathlib import Path

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
OUT = Path("out")
OUT.mkdir(exist_ok=True)

print(f"Device: {DEVICE} | PyTorch: {torch.__version__} | Transformers: {transformers.__version__}")

## Step 2. Models Registry & Shared Helpers / Étape 2. Modèles et fonctions partagées
Models are cached directly on the Google Colab VM disk (zero gigabytes downloaded onto your personal computer).

In [ ]:
import numpy as np
import soundfile as sf
from IPython.display import Audio, display
from transformers import AutoProcessor, MusicgenForConditionalGeneration, pipeline

MODELS = {
    "ast": "MIT/ast-finetuned-audioset-10-10-0.4593",
    "clap": "laion/clap-htsat-unfused",
    "whisper": "openai/whisper-tiny",
    "musicgen": "facebook/musicgen-small",
}

def make_demo_audio(sr=16000, seconds=5):
    """Fallback harmonic chord + percussive clicks if no mic/file is provided."""
    t = np.linspace(0, seconds, int(sr * seconds), endpoint=False)
    y = 0.25 * np.sin(2 * np.pi * 220 * t) + 0.2 * np.sin(2 * np.pi * 329.6 * t)
    y += 0.15 * np.sin(2 * np.pi * 440 * t) * np.sin(2 * np.pi * 2 * t)
    sf.write(OUT / "demo_input.wav", y.astype(np.float32), sr)
    return y.astype(np.float32)

y_demo = make_demo_audio(16000, 5)
display(Audio(y_demo, rate=16000))

## Step 3. Browser Microphone Capture in Colab / Étape 3. Microphone dans le navigateur
Records 5 seconds from your browser microphone directly into Colab (or falls back to `demo_input.wav`).

In [ ]:
import base64, io

def record_colab_mic(seconds=5, sr=16000):
    try:
        from google.colab import output
        js = f"""
        async function recordAudio() {{
          const stream = await navigator.mediaDevices.getUserMedia({{audio: true}});
          const rec = new MediaRecorder(stream);
          const chunks = [];
          rec.ondataavailable = e => chunks.push(e.data);
          rec.start();
          await new Promise(r => setTimeout(r, {seconds * 1000}));
          rec.stop();
          await new Promise(r => rec.onstop = r);
          stream.getTracks().forEach(t => t.stop());
          const blob = new Blob(chunks);
          const reader = new FileReader();
          reader.readAsDataURL(blob);
          return await new Promise(r => reader.onloadend = () => r(reader.result.split(',')[1]));
        }}
        recordAudio();
        """
        print(f"Recording {seconds}s from browser microphone...")
        b64 = output.eval_js(js)
        raw = base64.b64decode(b64)
        tmp_webm = OUT / "mic_raw.webm"
        tmp_webm.write_bytes(raw)
        y, _ = librosa.load(tmp_webm, sr=sr, mono=True)
        sf.write(OUT / "mic.wav", y, sr)
        return y
    except Exception as e:
        print("Using fallback demo audio:", e)
        return make_demo_audio(sr, seconds)

# Uncomment the next line to record live from your browser mic in Colab:
# y_input = record_colab_mic(seconds=5, sr=16000)
y_input = y_demo
display(Audio(y_input, rate=16000))

## Step 4. Inspect Audio & Mel-Spectrogram / Étape 4. Inspecter un fichier audio

In [ ]:
import matplotlib.pyplot as plt

S = librosa.power_to_db(librosa.feature.melspectrogram(y=y_input, sr=16000), ref=np.max)
plt.figure(figsize=(8, 3))
plt.imshow(S, origin="lower", aspect="auto", cmap="magma")
plt.title("Mel-Spectrogram (dB)")
plt.colorbar(format="%+2.0f dB")
plt.tight_layout()
plt.show()

## Step 5. Classify with AST (527 AudioSet labels) / Étape 5. Classer avec AST

In [ ]:
ast_clf = pipeline("audio-classification", model=MODELS["ast"], device=DEVICE)
preds = ast_clf({"raw": y_input, "sampling_rate": 16000}, top_k=5)
for p in preds:
    print(f"{p['label']:30s} : {p['score']:.3f}")

## Step 6. Zero-Shot Classification with CLAP / Étape 6. Zero-shot avec CLAP

In [ ]:
clap_clf = pipeline("zero-shot-audio-classification", model=MODELS["clap"], device=DEVICE)
labels = ["speech", "rain", "a synthesizer chord", "percussion", "silence"]
y_48k = librosa.resample(y_input, orig_sr=16000, target_sr=48000)
for r in clap_clf(y_48k, candidate_labels=labels):
    print(f"{r['label']:25s} : {r['score']:.3f}")

## Step 7. Speech-to-Text with Whisper / Étape 7. Parole vers texte avec Whisper

In [ ]:
asr = pipeline("automatic-speech-recognition", model=MODELS["whisper"], device=DEVICE)
text = asr({"raw": y_input, "sampling_rate": 16000})["text"]
print("Transcription:", text)

## Step 8. Text-to-Music with MusicGen / Étape 8. Texte vers musique

In [ ]:
processor = AutoProcessor.from_pretrained(MODELS["musicgen"])
musicgen = MusicgenForConditionalGeneration.from_pretrained(MODELS["musicgen"]).to(DEVICE)

def generate_music(prompt, seconds=6, audio=None):
    sr = musicgen.config.audio_encoder.sampling_rate
    kwargs = dict(text=[prompt], padding=True, return_tensors="pt")
    if audio is not None:
        kwargs.update(audio=np.asarray(audio, dtype=np.float32), sampling_rate=sr)
    inputs = processor(**kwargs).to(DEVICE)
    with torch.no_grad():
        out = musicgen.generate(**inputs, do_sample=True, guidance_scale=3.0, max_new_tokens=int(seconds * 50))
    return out[0, 0].cpu().numpy(), sr

wav_text, sr_mg = generate_music("slow ambient drone with bowed metal textures", seconds=6)
sf.write(OUT / "musicgen_text.wav", wav_text, sr_mg)
display(Audio(wav_text, rate=sr_mg))

## Step 9. Audio + Text Continuation / Étape 9. Texte et audio vers musique

In [ ]:
prompt_32k = librosa.resample(y_input, orig_sr=16000, target_sr=32000)
wav_cont, sr_mg = generate_music("sparse piano and granular noise", seconds=6, audio=prompt_32k)
sf.write(OUT / "musicgen_continuation.wav", wav_cont, sr_mg)
display(Audio(wav_cont, rate=sr_mg))

## Step 10. Minimal Agent Loop (Perceive → Decide → Act) / Étape 10. Boucle agentique
CLAP perceives the sound → a policy selects the next prompt → MusicGen generates the continuation, which feeds the next round.

In [ ]:
STYLES = {
    "speech": "sparse ambient pads with soft granular textures",
    "percussion": "dry polyrhythmic percussion, close microphone",
    "water": "slow harp arpeggios in a long reverb",
    "traffic": "industrial drones with metallic resonances",
    "silence": "low sustained drone, very slow evolution",
    "music": "contrasting counterpoint in a different timbre",
}

history = []
y_loop = librosa.resample(y_input, orig_sr=16000, target_sr=48000)

for i in range(3):
    best = clap_clf(y_loop, candidate_labels=list(STYLES))[0]
    label, score = best["label"], best["score"]
    prompt = STYLES[label]
    if history and history[-1] == label:
        prompt += ", with an abrupt change of register"
    history.append(label)
    print(f"Round {i}: heard '{label}' ({score:.2f}) -> prompt: '{prompt}'")

    ctx_32k = librosa.resample(y_loop, orig_sr=48000, target_sr=32000)[-5 * 32000:]
    out_wav, sr_mg = generate_music(prompt, seconds=6, audio=ctx_32k)
    display(Audio(out_wav, rate=sr_mg))
    y_loop = librosa.resample(out_wav, orig_sr=sr_mg, target_sr=48000)